# Ask 5 — Retrieve, Then Ask

The assembled system: retrieve top-k, build the contract prompt, ask.
Retrieval and prompt assembly run offline; the answering call uses the
class key (precomputed answers included).

In [ ]:
# The pile: documents from the (fictional) Jefferson High School.
# Real enough to search, small enough to read whole.
PILE = {
 "handbook_academics": """S4.1 Grading scale. A 90-100, B 80-89, C 70-79, D 60-69.
Semester grades weight exams at 30 percent.
S4.2 Exam Retake Policy. This policy applies to final exams only. Students
receive one retake per semester, requested within ten school days. The
higher score stands.
S4.3 Grade appeals. Appeals go to the department head in writing within
fifteen school days of the posted grade.
S4.5 Late work. Assignments lose 10 percent per school day late, to a
maximum of 50 percent. Teachers may grant extensions for documented
emergencies.""",
 "handbook_schedule": """S2.0 Bell schedule. Regular days run eight periods,
8:15 AM to 3:20 PM.
S2.1 Wednesday schedule. Dismissal at 1:30 PM every Wednesday for staff
development.
S2.4 Late arrival. Students arriving after 8:30 AM sign in at the main
office with a note.""",
 "handbook_trips": """S5.1 Field trips require a signed permission form
submitted five school days in advance.
S5.2 Trip costs above 20 dollars qualify for the student activity fund.
S5.4 Chaperones must be approved district volunteers.""",
 "handbook_athletics": """S6.2 Eligibility. Athletes must hold a C average
during their season. Freshmen may try out for varsity teams.
S6.3 Petitions. A varsity roster spot for a freshman requires a coach's
petition to the athletic director.""",
 "robotics_minutes": """Robotics club meets Tuesdays in room 214. Regional
trip is April 18; bring your signed permission form by April 10. Dues are
15 dollars for the year.""",
 "clubs_list": """Active clubs: robotics (Tuesdays), debate (Thursdays),
art collective (Fridays), chess (lunch, library). Sign-up forms at the
student office.""",
 "bus_routes": """Routes 12 and 15 serve the north side. Final pickup at
4:45 PM outside door C. Activity buses run Tuesday and Thursday only.""",
 "cafeteria": """Lunch periods run 11:10, 11:55, and 12:40. Breakfast is
served from 7:40 AM. Menus post monthly on the food services page.""",
}
print(f"{len(PILE)} documents, {sum(len(t) for t in PILE.values())} characters total")

In [ ]:
def chunk_by_section(pile, overlap_sentences=1):
    """Cut on the S-section seams; carry a sentence of overlap across cuts."""
    chunks = []
    for doc, text in pile.items():
        parts, current, header = [], [], None
        for line in text.splitlines():
            if line.strip().startswith("S") and len(line) > 2 and line.strip()[1].isdigit():
                if current:
                    parts.append((header, " ".join(current)))
                header, current = line.strip().split()[0].rstrip("."), [line]
            else:
                current.append(line)
        if current:
            parts.append((header, " ".join(current)))
        for i, (header, body) in enumerate(parts):
            text_out = body
            if overlap_sentences and i > 0:
                prev_tail = parts[i-1][1].split(". ")[-1]
                text_out = prev_tail + " ... " + body
            chunks.append({"doc": doc, "section": header or doc, "text": " ".join(text_out.split())})
    return chunks

CHUNKS = chunk_by_section(PILE)
print(f"{len(CHUNKS)} chunks")
for c in CHUNKS[:3]:
    print(f"  [{c['doc']} {c['section']}] {c['text'][:70]}...")

In [ ]:
import math, re, collections

def words(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2]

# document frequency: in how many chunks does each word appear?
DF = collections.Counter()
for c in CHUNKS:
    for w in set(words(c["text"])):
        DF[w] += 1

def score(query, chunk):
    """Shared words, each weighted by rarity: rare words shout, common words whisper."""
    shared = set(words(query)) & set(words(chunk["text"]))
    return sum(1.0 / DF[w] for w in shared)

def retrieve(query, k=3):
    ranked = sorted(CHUNKS, key=lambda c: -score(query, c))
    return ranked[:k]

print("retriever ready (word scorer; swap in embeddings from ask4 in Colab)")

In [ ]:
%pip install -q anthropic

In [ ]:
import os, getpass
# Ask your teacher for the class API key. getpass keeps it out of the file.
try:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Class API key: ")
    HAVE_KEY = len(os.environ["ANTHROPIC_API_KEY"]) > 10
except Exception:
    HAVE_KEY = False
print("Key loaded." if HAVE_KEY else "No key - precomputed outputs shown below each live cell.")

In [ ]:
MODEL = "claude-opus-5"

def llm(prompt, max_tokens=800):
    import anthropic
    client = anthropic.Anthropic()
    return client.messages.create(model=MODEL, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]).content[-1].text

## The contract prompt

Three clauses, each preventing a specific failure: no blending with
training memory, no uncited facts, no confident invention when the chunks
don't hold the answer.

In [ ]:
CONTRACT = """Answer using ONLY the sections provided below.
Cite the section id after each fact, like [S4.2].
If the sections do not contain the answer, reply exactly:
"That isn't in the provided documents." and say what the documents DO cover.

{sections}

Q: {question}"""

def build_prompt(question, k=3):
    chunks = retrieve(question, k)
    sections = "\n\n".join(f"[{c['section']}] ({c['doc']}) {c['text']}" for c in chunks)
    return CONTRACT.format(sections=sections, question=question), chunks

prompt, used = build_prompt("How many final exam retakes do I get?")
print(prompt)
assert "[S4.2]" in prompt and "ONLY" in prompt and "isn't in the provided" in prompt

## ask() — the whole system in one function

In [ ]:
def ask_pile(question, k=3):
    prompt, used = build_prompt(question, k)
    if HAVE_KEY:
        return llm(prompt), used
    return PRECOMPUTED.get(question, "(precomputed answer not written for this question)"), used

PRECOMPUTED = {
 "How many final exam retakes do I get?":
   "One retake per semester, for final exams only, requested within ten "
   "school days; the higher score stands [S4.2].",
 "What does the ski trip cost?":
   "That isn't in the provided documents. The provided sections cover exam "
   "retakes, grade appeals, and the late-work policy.",
}

for q in ["How many final exam retakes do I get?", "What does the ski trip cost?"]:
    answer, used = ask_pile(q)
    print("Q:", q)
    print("A:", answer)
    print("   (chunks provided:", [c["section"] for c in used], ")\n")

Note what the second answer did: the refusal clause fired, and the
answer honestly named what the retrieved chunks cover. That's the contract
working — lesson 6 is about catching the times it doesn't.

## Try it

1. Ask five questions with the class key. For each: right? cited? and for
   any refusal — correct refusal (truly absent) or retrieval miss (present,
   not found)? The receipts in `used` let you tell.
2. Remove the ONLY clause from the contract and re-ask the retake
   question. Compare details against S4.2 — anything blended in?
3. **Build turn-in:** your ten questions through `ask_pile`, graded on
   right / cited / refusal-type.